In [29]:
from conf import create_spark_session
from conf import config
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [ ]:
spark = create_spark_session()

In [31]:
df = spark.read.option("header", "true").csv(f"{config.root_path}{config.landing_zone}transfers.csv")

df.show()

+---------+-------------+---------------+------------+----------+--------------+---------------+------------+-------------------+----------------+
|player_id|transfer_date|transfer_season|from_club_id|to_club_id|from_club_name|   to_club_name|transfer_fee|market_value_in_eur|     player_name|
+---------+-------------+---------------+------------+----------+--------------+---------------+------------+-------------------+----------------+
|    16136|   2026-07-01|          26/27|         417|       123|      OGC Nice|        Retired|        NULL|         500000.000|           Dante|
|  1138758|   2026-07-01|          26/27|         336|       631|   Sporting CP|        Chelsea|52140000.000|       45000000.000|  Geovany Quenda|
|   195778|   2026-06-30|          25/26|          79|        27| VfB Stuttgart|  Bayern Munich|       0.000|       12000000.000| Alexander Nübel|
|   569033|   2026-06-30|          25/26|          39|        27|1.FSV Mainz 05|  Bayern Munich|       0.000|        4

In [32]:
df = df.withColumn("transfer_fee", F.col("transfer_fee").cast("int")) \
        .withColumn("market_value_in_eur", F.col("market_value_in_eur").cast("int")) \
        .withColumn("price_difference", F.col("transfer_fee") - F.col("market_value_in_eur")) \
        .withColumn("transfer_year", F.year("transfer_date")) \
        .filter(F.col("transfer_fee") > 0)

df.show()

+---------+-------------+---------------+------------+----------+--------------+--------------+------------+-------------------+--------------------+
|player_id|transfer_date|transfer_season|from_club_id|to_club_id|from_club_name|  to_club_name|transfer_fee|market_value_in_eur|         player_name|
+---------+-------------+---------------+------------+----------+--------------+--------------+------------+-------------------+--------------------+
|  1138758|   2026-07-01|          26/27|         336|       631|   Sporting CP|       Chelsea|    52140000|           45000000|      Geovany Quenda|
|   149729|   2025-07-01|          25/26|         294|       114|       Benfica|      Besiktas|     2000000|            2300000|          João Mário|
|   234811|   2025-07-01|          25/26|         150|       336|    Real Betis|   Sporting CP|     4700000|            6000000|           Rui Silva|
|   249303|   2025-07-01|          25/26|        3948|      3057|      Union SG|Standard Liège|     

In [33]:
# Regrouper par 'to_club_name' et calculer la moyenne pondérée
df_result = df.groupBy("to_club_name") \
    .agg(
        # Moyenne du surcout d'achat
        (F.sum(F.col("price_difference")) / F.count("to_club_name")).cast("int").alias("surcout_moyen"),
        # Total du surcout d'achat
        (F.sum(F.col("price_difference"))).alias("surcout_total"),
        # Compter le nombre de transferts par club
        F.count("to_club_name").alias("number_of_transfers")
    )

# Trier les résultats par 'market_value_in_eur' de la plus élevée à la moins élevée
df_sorted = df_result.orderBy(F.col("surcout_total").desc())

# Afficher les résultats
df_sorted.show()

[Stage 36:=============================>                            (1 + 1) / 2]

+-------------+-------------+-------------+-------------------+
| to_club_name|surcout_moyen|surcout_total|number_of_transfers|
+-------------+-------------+-------------+-------------------+
|      Chelsea|      9933896|    764910000|                 77|
|     Man City|      7448805|    499070000|                 67|
|      Man Utd|      9727400|    486370000|                 50|
|    Newcastle|      5492040|    269110000|                 49|
|      Arsenal|      4785600|    239280000|                 50|
|    Barcelona|      5730000|    217740000|                 38|
|     Juventus|      3167968|    202750000|                 64|
|     Brighton|      4014285|    196700000|                 49|
|      Everton|      4527325|    194675000|                 43|
|  Aston Villa|      3282033|    193640000|                 59|
|  Southampton|      3703846|    192600000|                 52|
|    Liverpool|      3825102|    187430000|                 49|
|    Tottenham|      3162586|    1834300

In [34]:
target_club = "Chelsea"

# Filtrer 'from_club_name' sur un club précis
df_filtered = df.filter(F.col("to_club_name") == target_club)

# Trier les résultats par 'transfer_fee' de la plus élevée à la moins élevée
df_sorted = df_filtered.orderBy(F.col("transfer_fee").desc())

# Afficher les premières lignes pour vérifier
df_sorted.show(50)

+---------+-------------+---------------+------------+----------+---------------+------------+------------+-------------------+--------------------+----------------+
|player_id|transfer_date|transfer_season|from_club_id|to_club_id| from_club_name|to_club_name|transfer_fee|market_value_in_eur|         player_name|price_difference|
+---------+-------------+---------------+------------+----------+---------------+------------+------------+-------------------+--------------------+----------------+
|   648195|   2023-01-31|          22/23|         294|       631|        Benfica|     Chelsea|   121000000|           55000000|      Enzo Fernández|        66000000|
|   687626|   2023-08-14|          23/24|        1237|       631|       Brighton|     Chelsea|   116000000|           75000000|      Moisés Caicedo|        41000000|
|    96341|   2021-08-12|          21/22|          46|       631|          Inter|     Chelsea|   113000000|          100000000|       Romelu Lukaku|        13000000|
|   

In [35]:
# Calculer les ventes : somme de transfer_fee lorsque le club est dans from_club_name
sales = (
    df.groupBy("from_club_name", "transfer_year")
    .agg(F.sum("transfer_fee").alias("total_sales"))
    .withColumnRenamed("from_club_name", "club_name")
)

# Calculer les achats : somme de transfer_fee lorsque le club est dans to_club_name
purchases = (
    df.groupBy("to_club_name", "transfer_year")
    .agg(F.sum("transfer_fee").alias("total_purchases"))
    .withColumnRenamed("to_club_name", "club_name")
)

# Fusionner les résultats : jointure sur club_name et transfer_year
result = sales.join(purchases, on=["club_name", "transfer_year"], how="outer") \
              .fillna(0, subset=["total_sales", "total_purchases"]) \
              .withColumn("balance", F.col("total_sales") - F.col("total_purchases"))

# Afficher les résultats
result.show()

[Stage 41:=============================>                            (1 + 1) / 2]

+--------------+-------------+-----------+---------------+---------+
|     club_name|transfer_year|total_sales|total_purchases|  balance|
+--------------+-------------+-----------+---------------+---------+
|1.FC K'lautern|         2011|          0|         530000|  -530000|
|1.FC K'lautern|         2012|    1500000|              0|  1500000|
|1.FC K'lautern|         2013|          0|         630000|  -630000|
|1.FC K'lautern|         2014|    3900000|        1000000|  2900000|
|1.FC K'lautern|         2015|    5500000|              0|  5500000|
|1.FC K'lautern|         2016|    1700000|              0|  1700000|
|1.FC K'lautern|         2017|    3500000|         100000|  3400000|
|1.FC K'lautern|         2020|    1550000|              0|  1550000|
|1.FC K'lautern|         2024|          0|          50000|   -50000|
|     1.FC Köln|         2010|          0|         300000|  -300000|
|     1.FC Köln|         2014|          0|        5300000| -5300000|
|     1.FC Köln|         2015|    

In [36]:
# Trouver le club avec la meilleure balance pour chaque année
window = Window.partitionBy("transfer_year").orderBy(F.col("balance").desc())

best_balances = (
    result.withColumn("rank", F.row_number().over(window))
          .filter(F.col("rank") == 1)
          .drop("rank")
          .orderBy("transfer_year")
)

# Afficher les meilleurs clubs par année
best_balances.show()

[Stage 49:>                                                         (0 + 1) / 1]

+-------------+-------------+-----------+---------------+---------+
|    club_name|transfer_year|total_sales|total_purchases|  balance|
+-------------+-------------+-----------+---------------+---------+
|    Barcelona|         2002|     750000|              0|   750000|
|        Leeds|         2004|    7400000|              0|  7400000|
|   Sevilla FC|         2005|   27000000|              0| 27000000|
|     Cobreloa|         2006|    3000000|              0|  3000000|
|     FC Porto|         2007|   30000000|         720000| 29280000|
|Dinamo Zagreb|         2008|   22500000|              0| 22500000|
|  Shakhtar D.|         2009|   25000000|              0| 25000000|
|        Inter|         2010|   29500000|              0| 29500000|
|VfL Wolfsburg|         2011|   37000000|              0| 37000000|
|      Benfica|         2012|   41200000|              0| 41200000|
|   Sevilla FC|         2013|   77000000|       14900000| 62100000|
|  Southampton|         2014|  114030000|       